# **Processing fluxomics & proteomics data from ECOMICS**

This notebook processes the ECOMICS dataset. 

ECOMICS is a comprehensive multi-omics resource for *Escherichia coli*, developed to enable predictive modeling of cellular states. It recovers 4,389 genome-wide profiles from 65 *E. coli* K-12 strains, 286 genetic perturbations, and 112 media conditions. 

The fluxomcis accounts for 43 profiles of 120 metabolic fluxes, and normalizes the values using the glucose uptake rate of each specific study. The fluxomics data can be found in `data/raw/ECOMICS/fluxomics_ecomics.csv`

The proteomics 

---
Kim, M., Rai, N., Zorraquino, V., & Tagkopoulos, I. (2016). Multi-omics integration accurately predicts cellular state in unexplored conditions for Escherichia coli. Nature Communications, 7(1). https://doi.org/10.1038/ncomms13090

## 1. Imports

In [13]:
%load_ext autoreload
%autoreload 2

import os
import sys

# Add paths
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../scripts'))

# Add directories
data_dir = "../data"

raw_data_dir = os.path.join(data_dir, "raw")
ecomics_raw_dir = os.path.join(raw_data_dir, "ECOMICS")

processed_data_dir = os.path.join(data_dir, "processed")
ecomics_processed_dir = os.path.join(processed_data_dir, "ECOMICS")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **Fluxomics**

## 2. Create a GEM reaction dataframe

In [14]:
from enzyme_classifier import create_gem_rxns_df

from cobra.io import load_model
import cobra 

model = load_model("iML1515")
df_rxns = create_gem_rxns_df(model)
processed_data_dir = os.path.join(data_dir, "processed")

#df_rxns.to_csv(ecomics_processed_dir, index=False)

## 3. Matching ECOMICS to iML1515 reactions

In [15]:
import pandas as pd
import cobra

# Load ECOMICS and iML1515 rxns dataframes
eco_path = os.path.join(raw_data_dir, "ECOMICS", "ecomics_rxns.csv")
eco = pd.read_csv(eco_path)

iml_path = os.path.join(ecomics_processed_dir, "iml1515_rxns.csv")
iml = pd.read_csv(iml_path)

# Manual correction of PDHm --> PDH
eco.loc[eco["bigg_id"] == "PDHm", "bigg_id"] = "PDH"

# Manual correction EX_succ(e) --> EX_succ_e
eco.loc[eco["bigg_id"] == "EX_succ(e)", "bigg_id"] = "EX_succ_e"

# Initialize list to map rxns between ecomics and gem
matches = []

for _, row in eco.iterrows():
    # Exact BiGG hit
    hit = iml.loc[iml["gem_bigg"] == row["bigg_id"]]

   # Empty row is no match
    if hit.empty:
        matches.append({
            "gem_rxn": pd.NA,
            "gem_rxn_id": pd.NA
        })
    else:
        h = hit.iloc[0]
        matches.append({
            "gem_rxn": h["gem_rxn"],
            "gem_rxn_id": h["gem_rxn_id"]
        })

# Final df that is the length of the ECOMICS rxns
eco_out = eco.join(pd.DataFrame(matches))
print(eco_out.shape)
print(eco_out.head())

(115, 5)
  ecomics_rxn_id                 ecomics_rxn   bigg_id  \
0          R0001  Glucose + PEP -> G6P + PYR  GLCptspp   
1          R0002                 G6P <-> F6P       PGI   
2          R0003                F6P -> F1,6P       PFK   
3          R0004         F1,6P -> DHAP + G3P       FBA   
4          R0005                 DHAP -> G3P       TPI   

                                 gem_rxn gem_rxn_id  
0     glc__D_p + pep_c --> g6p_c + pyr_c   GLCptspp  
1                        g6p_c <=> f6p_c        PGI  
2  atp_c + f6p_c --> adp_c + fdp_c + h_c        PFK  
3               fdp_c <=> dhap_c + g3p_c        FBA  
4                       dhap_c <=> g3p_c        TPI  


In [16]:
# Df with the successful BiGG matches
eco_out_hits = eco_out[eco_out['gem_rxn_id'].notna()]
print(eco_out_hits.shape)

(26, 5)


In [17]:
# Df with NO BiGG matches
eco_out_misses = eco_out[eco_out['gem_rxn_id'].isna()]

# Exporting for manual curation
eco_out_misses_csv = eco_out_misses.iloc[:, [0, 1]]
eco_out_misses_path = os.path.join(ecomics_processed_dir, "eco_out_misses.csv")
#eco_out_misses_csv.to_csv(eco_out_misses_path, index=False)

## 4. Manual curation of ECOMICS reactions

### Manual curation 

*Goal:* to map the reactions in the ECOMICS dataset without a BiGG ID. 

To do this, we'll look at each reaction in the unnanotated ECOMICS (**eco_out_misses**), and look for the best fit considering all of the reactions in the iML1515 GEM (**df_rxns**).

To make the process more efficient **eco_out_misses** and **df_rxns** were uploaded to NotebookLM, and prompted to find the best fit between these datasets.

Some mappings were direct, while others involved a combination of multiple sequential reactions into a single, more generalized reaction. Additionally, the level of detail differed, such as the inclusion of cofactors in the GEM reactions (e.g. ATP, ADP, NAD).

The final manually curated dataframe will have multiple iML1515 rxns mapped to a single ECOMICS rxn ID. Distributing this flux across the participant reactions remains pendant. Some ideas include:
- Choose a single best fitting reaction after the FBA simulations, where the GEM rxn with closest simulated flux to the experimental will be kept.
- Distribute the flux across all participating rxns (TBD how - possibly w a directed graph) 

In [18]:
# Manually curated dataframe
eco_curated_path = os.path.join(ecomics_processed_dir, "eco_out_misses_curated.csv")
eco_curated = pd.read_csv(eco_curated_path)

# Concatenate the curated df w the hits df
eco_curated_hits = pd.concat([eco_curated, eco_out_hits])

# Check the lenghts
print(f"eco curated: {eco_curated.shape}")
print(f"eco hits: {eco_out_hits.shape}")
print(f"eco curated + hits: {eco_curated_hits.shape}")

eco curated: (181, 5)
eco hits: (26, 5)
eco curated + hits: (207, 6)


In [19]:
# Order the rows using ecomics_rxn_id
eco_curated_hits = eco_curated_hits.sort_values(by='ecomics_rxn_id')

# Fill the bigg_id column using the gem_rxn_id
eco_curated_hits['bigg_id'] = eco_curated_hits['bigg_id'].fillna(eco_curated_hits['gem_rxn_id'])

# Create a mapping dictionary from df_rxns
rxn_mapping = df_rxns.set_index('gem_rxn_id')['gem_rxn'].to_dict()

# Fill the gem_rxn column using the mapping
eco_curated_hits['gem_rxn'] = eco_curated_hits['gem_rxn'].fillna(
    eco_curated_hits['gem_rxn_id'].map(rxn_mapping)
)

eco_curated_hits_missing = eco_curated_hits[eco_curated_hits['gem_rxn_id'].isna()]
print('Number reactions without a gem match: ', len(eco_curated_hits_missing))
print(f"eco curated + hits - missing: {eco_curated_hits.shape[0]-len(eco_curated_hits_missing)}")

Number reactions without a gem match:  19
eco curated + hits - missing: 188


In [20]:
# Load complete ECOMICS fluxomics dataframe
ecomicsfluxes_path = os.path.join(ecomics_raw_dir, "fluxomics_ecomics.csv")
ecomicsfluxes = pd.read_csv(ecomicsfluxes_path)

print(f"Original fluxomics shape: {ecomicsfluxes.shape}")
print('Total fluxes: ', ecomicsfluxes.shape[1]-4)
print('Total conditions: ', ecomicsfluxes.shape[0])
print(ecomicsfluxes.head())



Original fluxomics shape: (43, 124)
Total fluxes:  120
Total conditions:  43
    Strain MediumID         Medium Stress  R0001  R0002  R0003  R0004  R0005  \
0  BW25113    MD066  synthetic+Glu   none    100   63.0   79.0   79.0   79.0   
1  BW25113    MD066  synthetic+Glu   none    100   62.0   80.0   80.0   80.0   
2    W3110    MD121         M9+Glu   none    100   73.0   83.1   83.1   83.1   
3  BW25113    MD066  synthetic+Glu   none    100   78.0   79.0   79.0   79.0   
4  BW25113    MD066  synthetic+Glu   none    100   63.0   80.0   80.0   80.0   

   R0006  ...  R0111  R0112  R0113  R0114  R0115  R0116  R0117  R0118  R0119  \
0  165.0  ...    0.0    0.0      0    0.0    0.0    0.0    0.0    0.0    0.0   
1  167.0  ...    0.0    0.0      0    0.0    0.0    0.0    0.0    0.0    0.0   
2  171.5  ...   71.2   69.0      0  113.4   27.2   10.1    7.5    9.0    0.9   
3  171.0  ...    0.0    0.0      0    0.0    0.0    0.0    0.0    0.0    0.0   
4  168.0  ...    0.0    0.0      0    0.0 

In [21]:
from collections import defaultdict

# Select first 4 metadata cols
metadata_cols = ecomicsfluxes.columns[:4]

# Create extended reaction names in eco_curated_hits
eco_curated_hits['extended_rxn_id'] = eco_curated_hits['ecomics_rxn_id'] + '_' + eco_curated_hits['gem_rxn_id']

# Create a mapping from original ecomics_rxn_id to list of extended reaction names
ecomics_to_extended = defaultdict(list)

for _, row in eco_curated_hits.iterrows():
    ecomics_to_extended[row['ecomics_rxn_id']].append({
        'extended_name': row['extended_rxn_id'],
        'gem_rxn_id': row['gem_rxn_id']
    })

# Find reaction columns that have mappings
matched_rxn_cols = [col for col in ecomicsfluxes.columns if col in ecomics_to_extended]

# Create expanded flux dataframe
expanded_flux_data = {}

# Copy original flux values to extended reaction columns
for ecomics_rxn in matched_rxn_cols:
    original_flux_values = ecomicsfluxes[ecomics_rxn]
    
    # For each extended reaction mapping
    for mapping in ecomics_to_extended[ecomics_rxn]:
        extended_name = mapping['extended_name']
        # Duplicate the original flux values
        expanded_flux_data[extended_name] = original_flux_values

# Create the expanded dataframe
expanded_fluxes_df = pd.DataFrame(expanded_flux_data)
print(expanded_fluxes_df.shape)
print(expanded_fluxes_df.head())

(43, 184)
   R0001_GLCptspp  R0002_PGI  R0003_PFK  R0004_FBA  R0005_TPI  R0006_GAPD  \
0             100       63.0       79.0       79.0       79.0       165.0   
1             100       62.0       80.0       80.0       80.0       167.0   
2             100       73.0       83.1       83.1       83.1       171.5   
3             100       78.0       79.0       79.0       79.0       171.0   
4             100       63.0       80.0       80.0       80.0       168.0   

   R0007_ENO  R0007_PGM  R0008_PYK  R0009_PDH  ...  R0112_ACONTa  \
0      153.0      153.0       25.0      101.0  ...           0.0   
1      157.0      157.0       53.0      136.0  ...           0.0   
2      159.0      159.0       27.1      115.7  ...          69.0   
3      164.0      164.0       48.0      147.0  ...           0.0   
4      158.0      158.0       54.0      144.0  ...           0.0   

   R0112_ACONTb  R0113_ICDHyr  R0114_SUCCOAS  R0115_SUCDi  R0116_ICL  \
0           0.0             0            0.0  

In [22]:
# Combine metadata and expanded fluxes
ecomicsfluxes_mapped = pd.concat([ecomicsfluxes[metadata_cols], expanded_fluxes_df], axis=1)

print('ECOMICS fluxes: ', ecomicsfluxes.shape[1]-4)
print(f"ECOMICS curated + hits - missing: {eco_curated_hits.shape[0]-len(eco_curated_hits_missing)}")
print(f"Fluxes mapped: {len(expanded_flux_data)}")

#ecomicsfluxes_mapped.to_csv(os.path.join(ecomics_processed_dir, "fluxomics_ecomics_mapped.csv"), index=False)

ECOMICS fluxes:  120
ECOMICS curated + hits - missing: 188
Fluxes mapped: 184


### TO DO: There are 4 missing reactions in expanded_fluxes_df. Its lenght is 184 and it should be 188 (eco curated + hits - missing: 188)

## 5. Inspect most complete studies in ECOMICS

In [23]:
from scripts.ECOMICS_analysis import select_most_complete_study

# Check the most complete study - with the most non-zero fluxes
completeness_df = select_most_complete_study(ecomicsfluxes_mapped)
completeness_df = completeness_df.sort_values(by='num_nonzero_rxns', ascending=False)
print(completeness_df.head())

     Strain MediumID         Medium Stress  num_nonzero_rxns  R0001_GLCptspp  \
2     W3110    MD121         M9+Glu   none               161             100   
39    W3110    MD121         M9+Glu   none               161             100   
20    W3110    MD121         M9+Glu   none               161             100   
14  BW25113    MD066  synthetic+Glu   none                39             100   
22  BW25113    MD066  synthetic+Glu   none                39             100   

    R0002_PGI  R0003_PFK  R0004_FBA  R0005_TPI  ...  R0112_ACONTa  \
2        73.0       83.1       83.1       83.1  ...          69.0   
39       77.5       84.0       84.0       84.0  ...           5.6   
20       73.0       83.4       83.4       83.4  ...           4.7   
14       67.0       69.0       69.0       69.0  ...           0.0   
22        0.0       34.0       34.0       34.0  ...           0.0   

    R0112_ACONTb  R0113_ICDHyr  R0114_SUCCOAS  R0115_SUCDi  R0116_ICL  \
2           69.0             0 

### *TO DO:* Check if the 43 studies can be averaged (across the same conditions) or must be kept independent
For now we'll be working with study index 2

In [24]:
exp_fluxomics_df = ecomicsfluxes_mapped.iloc[2].to_frame('exp_flux')
condition_string = '_'.join(exp_fluxomics_df.iloc[:4, 0].astype(str).values) 
exp_fluxomics_df = exp_fluxomics_df.iloc[4:] # discard the first 4 rows (metadata)
exp_fluxomics_df = exp_fluxomics_df.reset_index().rename(columns={'index': 'exp_reaction'})
#exp_fluxomics_df.to_csv(os.path.join(ecomics_processed_dir, f"fluxomics_{condition_string}.csv"), index=False)

print(f'Created fluxomics dataframe for {condition_string}')


Created fluxomics dataframe for W3110_MD121_M9+Glu_none


## 6. Apply ECOMICS conditions to the iML1515 GEM

In [25]:
from scripts.ECOMICS_analysis import apply_ecomics_condition

modified_model = apply_ecomics_condition("MD121", "none")

Applying M9+Glu medium
Set EX_glc__D_e lower bound to -10


## 7. Comparison of fluxomics: ECOMICS vs FBA vs pFBA

### *TO DOs prior to this step*:
- Include exchange flux values into model medium
- Reconcile ECOMICS fluxes to match an S matrix

In [26]:
from scripts.ECOMICS_analysis import create_fluxomics_dataframe

condition_string = 'W3110_MD121_M9+Glu_none'
exp_fluxes_path = os.path.join(ecomics_processed_dir, f"fluxomics_{condition_string}.csv")

fba_df, pFBA_df, fluxomics_df = create_fluxomics_dataframe(exp_fluxes_path, modified_model)

FBA status: optimal
pFBA status: optimal
Multiple experimental reactions found for SHK3Dr: ['R0091_SHK3Dr', 'R0092_SHK3Dr']
Multiple experimental reactions found for ICDHyr: ['R0019_ICDHyr', 'R0113_ICDHyr']
Multiple experimental reactions found for PPCK: ['R0050_PPCK', 'R0070_PPCK']
Multiple experimental reactions found for EX_co2_e: ['R0043_EX_co2_e', 'R0102_EX_co2_e']
Multiple experimental reactions found for PSCVT: ['R0091_PSCVT', 'R0092_PSCVT']
Multiple experimental reactions found for CHORS: ['R0091_CHORS', 'R0092_CHORS']
Multiple experimental reactions found for ENO: ['R0007_ENO', 'R0045_ENO']
Multiple experimental reactions found for FBA: ['R0004_FBA', 'R0059_FBA', 'R0120_FBA']
Multiple experimental reactions found for PGI: ['R0002_PGI', 'R0044_PGI']
Multiple experimental reactions found for EDA: ['R0028_EDA', 'R0069_EDA']
Multiple experimental reactions found for DHQTi: ['R0091_DHQTi', 'R0092_DHQTi']
Multiple experimental reactions found for TALA: ['R0015_TALA', 'R0047_TALA', '

### *TO DOs:* 
- There is a NaN reaction, revise the mapping in Step 4 (cell 12)
- Modify the selection for the 'Multiple experimental reactions'. Keep the experimental flux that is closest to the simulations?

In [27]:
# scale the experimental fluxes column by the glucose uptake rate
fluxomics_df['exp_flux'] = fluxomics_df['exp_flux'] / 10

# **Proteomics**

## 8. Create enzymes dataframe

In [28]:
# Module imports
from scripts.enzyme_classifier import create_gpr_dataframe, analyze_model_gprs

from cobra.io import load_model
import cobra 

# Load the model
model = load_model("iML1515")

# Create a dataframe of the GPR rules
df_enzymes = create_gpr_dataframe(model)

# Analyze the model
stats = analyze_model_gprs(model)
print(f"\nModel Stats:")
print(f"Total reactions: {stats['total_reactions']}")
print(f"Reactions with GPR: {stats['reactions_with_gpr']}")
print(f"Total genes: {stats['total_genes']}")
print(f"GPR cases: {stats['gpr_complexity']}")


Model Stats:
Total reactions: 2712
Reactions with GPR: 2266
Total genes: 1516
GPR cases: {'simple': 1302, 'or_only': 651, 'and_only': 221, 'complex': 92}


In [29]:
# Dataframe of homomeric enzymes
df_homomeric = df_enzymes[df_enzymes['type'] == 'homomeric']

# Dataframe of unique homomeric enzymes 
df_homomeric_unique = df_homomeric.drop_duplicates(subset='gene') # TO CHECK - which one to use?

## 9. Map the homomeric enzyme genes to each reaction

In [30]:
# Create a mapping from reaction to homomeric gene
homomeric_mapping = df_homomeric.set_index('rxn')['gene'].to_dict()

fluxomics_genes_df = fluxomics_df.copy()

# Map the genes to fluxomics_df
fluxomics_genes_df['homomeric_gene'] = fluxomics_genes_df['rxn_id'].map(homomeric_mapping)

## 10. Add sequence information

In [31]:
# Read sequences recovered from UniProt
seq_path = os.path.join(processed_data_dir, 'UniProt', 'iML1515_E coli_83333_UniProt.csv')

seq_df = pd.read_csv(seq_path)

# Add sequence column matching the genes
seq_mapping = seq_df.set_index('model_gene_id')['sequence'].to_dict()

fluxomics_genes_df['sequence'] = fluxomics_genes_df['homomeric_gene'].map(seq_mapping)

## 11. Map proteomics using gene IDs

*TO DO*
- Check if there is matching condition to the one used for the fluxomics
- Read on the normalization done to the proteomics (what are the units?)